In [1]:
import pandas as pd
import math as m
from scipy.stats import norm
from itertools import product
from time import sleep

In [2]:
test_results = pd.read_csv("test_results.csv")

In [3]:
test_results

,section,version,survey answer,is correct,count
0,1,1,used,True,19
1,1,1,used,False,12
2,1,1,not used,True,10
3,1,1,not used,False,4
4,1,1,not aware,True,0
5,1,1,not aware,False,1
6,1,1,didn’t answer,True,3
7,1,1,didn’t answer,False,2
8,1,2,used,True,13
9,1,2,used,False,12


In [90]:
def same_distro(successes1, trials1, successes2, trials2, alter_hypoth="two-sided"):
    x1, n1, x2, n2 = successes1, trials1, successes2, trials2
    # Sample proportions
    p1 = x1 / n1
    p2 = x2 / n2
    
    # Pooled proportion under H0: p1 = p2
    p_pool = (x1 + x2) / (n1 + n2)
    
    # Standard error
    se = m.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
    
    # Z statistic
    z = (p1 - p2) / se

    # null hypothesis is p_1 = p_2
    # P-value based on alternative hypothesis
    if alter_hypoth == "two-sided": # H1: p1 != p2
        p_value = 2 * (1 - norm.cdf(abs(z)))
    elif alter_hypoth == "greater":   # H1: p1 > p2
        p_value = 1 - norm.cdf(z)
    elif alter_hypoth == "less":      # H1: p1 < p2
        p_value = norm.cdf(z)
    else:
        raise ValueError("alternative must be 'two-sided', 'greater', or 'less'")
    
    return z, p_value

In [103]:
sections = [1, 2]
versions = [1, 2]
# survey_answers = ["used", "not used", "not aware", "didn’t answer"]
survey_answers = ["used", "not used"]
corrects = [True, False]
p_values = pd.DataFrame([], columns=["sec1", "ver1", "ans1", "x1", "n1", "sec2", "ver2", "ans2", "x2", "n2", "p-val"])
for section1, section2, version1, version2, survey_answer1, survey_answer2 in \
        product(sections, sections, versions, versions, survey_answers, survey_answers):
    # print(f"{section1=} {section2=} {version1=} {version2=} {survey_answer1=} {survey_answer2=}")
    if section1 > section2 or version1 > version2 or survey_answers.index(survey_answer1) > survey_answers.index(survey_answer2):
        continue
    if section1 == section2 and version1 == version2 and survey_answer1 == survey_answer2:
        continue
    successes1 = test_results[
        (test_results["section"] == section1) &
        (test_results["version"] == version1) &
        (test_results["survey answer"] == survey_answer1) &
        (test_results["is correct"] == True)
    ]["count"].iloc[0]
    trials1 = sum(
        test_results[
            (test_results["section"] == section1) &
            (test_results["version"] == version1) &
            (test_results["survey answer"] == survey_answer1)
        ]["count"]
    )
    successes2 = test_results[
        (test_results["section"] == section2) &
        (test_results["version"] == version2) &
        (test_results["survey answer"] == survey_answer2) &
        (test_results["is correct"] == True)
    ]["count"].iloc[0]
    trials2 = sum(
        test_results[
            (test_results["section"] == section2) &
            (test_results["version"] == version2) &
            (test_results["survey answer"] == survey_answer2)
        ]["count"]
    )
    # sleep(0.1)
    # print(f"{section1=} {version1=} {survey_answer1=} {trials1=}")
    # print(f"{section2=} {version2=} {survey_answer2=} {trials2=}")
    _, p_value = same_distro(successes1, trials1, successes2, trials2)
    # sleep(0.1)
    survey_answer1 = "no answer" if survey_answer1 == "didn’t answer" else survey_answer1
    survey_answer2 = "no answer" if survey_answer2 == "didn’t answer" else survey_answer2
    if True: # p_value > 0.8:
        p_values.loc[len(p_values)] = [
            section1,
            version1,
            survey_answer1,
            successes1,
            trials1,
            section2,
            version2,
            survey_answer2,
            successes2,
            trials2,
            p_value
        ]
p_values

,sec1,ver1,ans1,x1,n1,sec2,ver2,ans2,x2,n2,p-val
0,1,1,used,19,31,1,1,not used,10,14,0.510704
1,1,1,used,19,31,1,2,used,13,25,0.484936
2,1,1,used,19,31,1,2,not used,11,14,0.254930
3,1,1,not used,10,14,1,2,not used,11,14,0.662521
4,1,2,used,13,25,1,2,not used,11,14,0.101803
5,1,1,used,19,31,2,1,used,17,26,0.749597
6,1,1,used,19,31,2,1,not used,15,21,0.450841
7,1,1,not used,10,14,2,1,not used,15,21,1.000000
8,1,1,used,19,31,2,2,used,16,27,0.874673
9,1,1,used,19,31,2,2,not used,8,12,0.743552


In [104]:
sections = [1, 2]
versions = [1, 2]
survey_answers = ["used", "not used", "not aware", "didn’t answer"]
# survey_answers = ["used", "not used"]
corrects = [True, False]
p_values = pd.DataFrame([], columns=["ans1", "x1", "n1", "ans2", "x2", "n2", "p-val"])
for survey_answer1, survey_answer2 in product(survey_answers, survey_answers):
    # print(f"{section1=} {section2=} {version1=} {version2=} {survey_answer1=} {survey_answer2=}")
    # if section1 > section2 or version1 > version2 or survey_answers.index(survey_answer1) > survey_answers.index(survey_answer2):
    #     continue
    if section1 == section2 and version1 == version2 and survey_answer1 == survey_answer2:
        continue
    successes1 = sum(test_results[
        # (test_results["section"] == section1) &
        # (test_results["version"] == version1) &
        (test_results["survey answer"] == survey_answer1) &
        (test_results["is correct"] == True)
    ]["count"])
    trials1 = sum(
        test_results[
            # (test_results["section"] == section1) &
            # (test_results["version"] == version1) &
            (test_results["survey answer"] == survey_answer1)
        ]["count"]
    )
    successes2 = sum(test_results[
        # (test_results["section"] == section2) &
        # (test_results["version"] == version2) &
        (test_results["survey answer"] == survey_answer2) &
        (test_results["is correct"] == True)
    ]["count"])
    trials2 = sum(
        test_results[
            # (test_results["section"] == section2) &
            # (test_results["version"] == version2) &
            (test_results["survey answer"] == survey_answer2)
        ]["count"]
    )
    # sleep(0.1)
    # print(f"{section1=} {version1=} {survey_answer1=} {trials1=}")
    # print(f"{section2=} {version2=} {survey_answer2=} {trials2=}")
    _, p_value = same_distro(successes1, trials1, successes2, trials2, "less")
    # sleep(0.1)
    survey_answer1 = "no answer" if survey_answer1 == "didn’t answer" else survey_answer1
    survey_answer2 = "no answer" if survey_answer2 == "didn’t answer" else survey_answer2
    if p_value < 0.1:
        p_values.loc[len(p_values)] = [
            survey_answer1,
            successes1,
            trials1,
            survey_answer2,
            successes2,
            trials2,
            p_value
        ]
p_values

,ans1,x1,n1,ans2,x2,n2,p-val
0,used,65,109,not used,44,61,0.051598
1,not aware,7,16,not used,44,61,0.016314
2,no answer,7,14,not used,44,61,0.054696


In [42]:
raw_survey_results = pd.read_csv("survey_results.csv")

In [43]:
raw_survey_results

,Timestamp,User ID from the lambda calculus website\n(automatically filled in),Have you taken this interactive lesson in lambda calculus?\n(automatically filled in),How well do you know lambda calculus?,What does the following expression reduce to after one beta reduction (with renaming if needed)?\n(λx. x (λx. y x)) z,How many redexes are in the following expression:\n( λx.λz. ( λz.y ) x ) ( w ( λw.y ) λx.w ),What is the parameter in the following expression:\n(λa.b)c,Does the following expression need to have a variable(s) renamed before it can be reduced:\n(λy.λv.λw.v y) λv.v y,Do you have any general comments or feedback that you wish to share?
0,3/26/2026 14:21:17,b43b391a,No,I have heard of lambda calculus,z (λx. y x),2,c,No,NaN
1,4/2/2026 14:04:40,b43b391a,No,I know it very well,z (λx. y x),2,a,No,NaN
2,4/8/2026 11:32:27,nolan,No,I know it very well,z (λx. y x),0,b,Yes,NaN
3,4/8/2026 13:21:53,b43b3954,No,I have limited familiarity with lambda calculus,z (λx. y x),2,c,Yes,NaN
4,4/8/2026 13:21:53,b43b3956,No,I am familiar with lambda calculus,x (λx. y x),3,b,Yes,NaN
...,...,...,...,...,...,...,...,...,...
71,4/10/2026 14:09:30,b43b39f,No,I am familiar with lambda calculus,z (λx. y x),2,(λa.b),Yes,NaN
72,4/11/2026 15:37:46,b43b3a03,No,I have heard of lambda calculus,x (λx. y x),4,c,Yes,I need to get a lesson from you.
73,4/12/2026 10:42:08,b43b3a09,Yes,I have heard of lambda calculus,z (λx. y x),2,a,Yes,I have a long way to go but the lesson seemed ...
74,4/13/2026 17:39:55,b43b3a07,No,I am familiar with lambda calculus,z (λx. y x),1,a,Yes,NaN


In [47]:
survey_results = pd.DataFrame([], columns=[
    "user id", "is before", "knowedge level", "reduction question", "redex count question",
    "parameter question", "need renaming question"
])
rows = [raw_survey_results.loc[i] for i in range(len(raw_survey_results))]
rows.sort(key=lambda row: row["User ID from the lambda calculus website\n(automatically filled in)"])
for i, row in enumerate(rows):
    survey_results.loc[len(survey_results)] = [
        row["User ID from the lambda calculus website\n(automatically filled in)"],
        row["Have you taken this interactive lesson in lambda calculus?\n(automatically filled in)"] == "No",
        row["How well do you know lambda calculus?"],
        row["What does the following expression reduce to after one beta reduction (with renaming if needed)?\n(λx. x (λx. y x)) z"]
            == "z (λx. y x)",
        row["How many redexes are in the following expression:\n( λx.λz. ( λz.y ) x ) ( w ( λw.y ) λx.w )"] == 2,
        row["What is the parameter in the following expression:\n(λa.b)c"] == "b",
        row["Does the following expression need to have a variable(s) renamed before it can be reduced:\n(λy.λv.λw.v y) λv.v y"] == "Yes"
    ]
for user_id in set(survey_results["user id"]):
    is_before = list(survey_results[survey_results["user id"] == user_id]["is before"])
    if True in is_before and False in is_before:
        continue
    survey_results = survey_results[survey_results["user id"] != user_id]
survey_results.reset_index(drop=True, inplace=True)
survey_results

,user id,is before,knowedge level,reduction question,redex count question,parameter question,need renaming question
0,b43b3961,True,I am familiar with lambda calculus,True,True,False,False
1,b43b3961,False,I am familiar with lambda calculus,True,True,False,False
2,b43b3963,True,I am familiar with lambda calculus,True,False,False,False
3,b43b3963,False,I am familiar with lambda calculus,True,True,False,False
4,b43b3965,True,I have limited familiarity with lambda calculus,True,False,True,True
5,b43b3965,False,I have limited familiarity with lambda calculus,False,True,False,True
6,b43b397b,True,I am familiar with lambda calculus,True,True,False,True
7,b43b397b,False,I know it very well,True,True,False,True
8,b43b397d,True,I am familiar with lambda calculus,True,False,False,False
9,b43b397d,False,I am familiar with lambda calculus,True,False,False,False


In [102]:
knowedge_levels = [
    "I know it very well",
    "I am familiar with lambda calculus",
    "I have limited familiarity with lambda calculus",
    "I have heard of lambda calculus",
    "I do not know what lambda calculus is",
    "all knowedge levels"
]
questions = ["reduction question", "redex count question", "parameter question", "need renaming question"]
p_values = pd.DataFrame([], columns=["question", "knowedge level", "before", "after", "total", "p-val"])
for knowedge_level, question in product(knowedge_levels, questions):
    # print(f"{section1=} {section2=} {version1=} {version2=} {survey_answer1=} {survey_answer2=}")
    # if section1 > section2 or version1 > version2 or survey_answers.index(survey_answer1) > survey_answers.index(survey_answer2):
    #     continue
    user_ids = survey_results[((survey_results["knowedge level"] == knowedge_level) | ("all knowedge levels" == knowedge_level))]["user id"]
    total = len(survey_results[
        (survey_results["is before"] == True) &
        (survey_results["user id"].isin(user_ids))
    ])
    if total == 0:
        continue
    before = len(survey_results[
        (survey_results["is before"] == True) &
        (survey_results["user id"].isin(user_ids)) &
        (survey_results[question] == True)
    ])
    after = len(survey_results[
        (survey_results["is before"] == False) &
        (survey_results["user id"].isin(user_ids)) &
        (survey_results[question] == True)
    ])
    # print((survey_results["user id"].isin(user_ids)))
    # print(survey_results[(survey_results["user id"].isin(user_ids))])
    print((before, total, after, total, "less"))
    if before == after and (before == total or before == 0):
        p_value = 1.
    else:
        _, p_value = same_distro(before, total, after, total, "less")
    if True: # total >= 3 and p_value < 0.1: # or True:
        p_values.loc[len(p_values)] = [question, knowedge_level, before, after, total, p_value]
p_values

(2, 2, 2, 2, 'less')
(2, 2, 2, 2, 'less')
(0, 2, 0, 2, 'less')
(1, 2, 1, 2, 'less')
(13, 13, 13, 13, 'less')
(6, 13, 9, 13, 'less')
(1, 13, 1, 13, 'less')
(7, 13, 7, 13, 'less')
(5, 5, 4, 5, 'less')
(2, 5, 3, 5, 'less')
(1, 5, 0, 5, 'less')
(3, 5, 4, 5, 'less')
(1, 1, 1, 1, 'less')
(0, 1, 1, 1, 'less')
(0, 1, 0, 1, 'less')
(1, 1, 0, 1, 'less')
(15, 15, 14, 15, 'less')
(7, 15, 11, 15, 'less')
(2, 15, 1, 15, 'less')
(8, 15, 8, 15, 'less')


,question,knowedge level,before,after,total,p-val
0,reduction question,I know it very well,2,2,2,1.000000
1,redex count question,I know it very well,2,2,2,1.000000
2,parameter question,I know it very well,0,0,2,1.000000
3,need renaming question,I know it very well,1,1,2,0.500000
4,reduction question,I am familiar with lambda calculus,13,13,13,1.000000
5,redex count question,I am familiar with lambda calculus,6,9,13,0.116851
6,parameter question,I am familiar with lambda calculus,1,1,13,0.500000
7,need renaming question,I am familiar with lambda calculus,7,7,13,0.500000
8,reduction question,I have limited familiarity with lambda calculus,5,4,5,0.854080
9,redex count question,I have limited familiarity with lambda calculus,2,3,5,0.263545
